# 01 — Data Loading & Initial Inspection

**Fase PACE**: Plan  
**Obiettivo**: caricare i due dataset LAPD (2010–2019 e 2020–2024), verificarne la struttura, allineare le colonne, concatenarli e produrre un'ispezione iniziale che guidi le decisioni della fase Analyze.

**Output**: `data/processed/crimes_merged.parquet` — dataset combinato non ancora pulito, con tipi di dato consistenti.

## 1. Setup

In [29]:
import pandas as pd

In [30]:
pd.set_option('display.max_rows',None)          # Mosta tutte le righe

pd.set_option('display.max_columns',None)       # Mostra tutte le colonne

pd.set_option('display.max_info_columns',200)   # df.info() mostra 200 colonne
                                                # (200 è un numero indicativo, che serve per mostrare tutte le colonne
                                                #  se le colonne fossero di più di 200 bisognerebbe inserire un numero maggiore)

## 2. Caricamento dei dataset grezzi

In [31]:
df1 = pd.read_csv('../../data/raw/crime-2010-2019.csv') # Lettura DataFrame relativo ai crimini 2010-2019
df2 = pd.read_csv('../../data/raw/crime-2020-2024.csv') # Lettura DataFrame relativo ai crimini 2020-2024

## 3. Verifica e allineamento delle colonne

Confrontiamo i nomi delle colonne dei due DataFrame prima della concatenazione.

In [32]:
var1=df1.columns # Creazione della lista delle colonne dei DataFrame 'df1' e 'df2'
var2=df2.columns # per controllare se il nome delle colonne sia lo stesso per entrambi i DataFrame prima del merge

print( f'Le colonne del primo df sono: {var1}')
print( f'Le colonne del secondo df sono: {var2}')

Le colonne del primo df sono: Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA ', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='object')
Le colonne del secondo df sono: Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='object')


In [33]:
var1==var2  # Confronto del nome delle colonne

array([ True,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

### Problema rilevato

Il confronto rivela una differenza: nel dataset 2010–2019 la colonna `AREA` ha uno spazio finale (`'AREA '`), mentre nel 2020–2024 no.

**Soluzione**: applichiamo `.str.strip()` ai nomi colonna di entrambi i DataFrame per normalizzare eventuali spazi e garantire l'allineamento.

In [34]:
df1.columns = df1.columns.str.strip() 
df2.columns = df2.columns.str.strip()

In [35]:
new_col1 = df1.columns
new_col2 = df2.columns

new_col1==new_col2

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

## 4. Concatenazione dei due dataset

In [36]:
df = pd.concat([df1, df2], ignore_index=True)
print(df.shape)
print(df.info())

(3138031, 28)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3138031 entries, 0 to 3138030
Data columns (total 28 columns):
 #   Column          Dtype  
---  ------          -----  
 0   DR_NO           int64  
 1   Date Rptd       object 
 2   DATE OCC        object 
 3   TIME OCC        int64  
 4   AREA            int64  
 5   AREA NAME       object 
 6   Rpt Dist No     int64  
 7   Part 1-2        int64  
 8   Crm Cd          int64  
 9   Crm Cd Desc     object 
 10  Mocodes         object 
 11  Vict Age        int64  
 12  Vict Sex        object 
 13  Vict Descent    object 
 14  Premis Cd       float64
 15  Premis Desc     object 
 16  Weapon Used Cd  float64
 17  Weapon Desc     object 
 18  Status          object 
 19  Status Desc     object 
 20  Crm Cd 1        float64
 21  Crm Cd 2        float64
 22  Crm Cd 3        float64
 23  Crm Cd 4        float64
 24  LOCATION        object 
 25  Cross Street    object 
 26  LAT             object 
 27  LON             object 
dty

## 5. Ispezione iniziale

### 5.1 Duplicati su DR_NO

In [37]:
duplicati = df['DR_NO'].duplicated().sum()
print(f"Duplicati su DR_NO: {duplicati}")

Duplicati su DR_NO: 57809


### 5.2 Valori nulli per colonna

In [38]:
nulli_pct = (df.isnull().sum() / len(df) * 100).round(2)
nulli_pct = nulli_pct[nulli_pct > 0].sort_values(ascending=False)
print("Percentuale di valori nulli per colonna:")
print(nulli_pct)

Percentuale di valori nulli per colonna:
Crm Cd 4          99.99
Crm Cd 3          99.81
Crm Cd 2          93.27
Cross Street      83.71
Weapon Used Cd    66.73
Weapon Desc       66.73
Mocodes           12.15
Vict Sex          10.92
Vict Descent      10.92
Premis Desc        0.02
dtype: float64


### 5.3 Distribuzione record per anno

In [39]:
df['anno_temp'] = pd.to_datetime(df['DATE OCC'], errors='coerce').dt.year
print("Record per anno:")
print(df['anno_temp'].value_counts().sort_index())
df.drop(columns='anno_temp', inplace=True)

/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_4164/4075905120.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['anno_temp'] = pd.to_datetime(df['DATE OCC'], errors='coerce').dt.year


Record per anno:
anno_temp
2010    209325
2011    200912
2012    201835
2013    192875
2014    195879
2015    168076
2016    283798
2017    231751
2018    229768
2019    218918
2020    199847
2021    209876
2022    235259
2023    232345
2024    127567
Name: count, dtype: int64


## 6. Normalizzazione LAT/LON

### Problema rilevato

Dall'ispezione iniziale è emerso che le colonne `LAT` e `LON` sono di tipo `object` invece che `float64`. Indagando i due dataset separatamente è emersa la causa:

- **Dataset 2010–2019**: usa la **virgola** come separatore decimale (es. `33,9825`) — formato europeo
- **Dataset 2020–2024**: usa il **punto** come separatore decimale (es. `34.2124`) — formato standard anglosassone

Pandas inferisce quindi tipi diversi: `float64` per il secondo dataset, `object` (stringa) per il primo, perché non riconosce `33,9825` come numero valido.

### Tentativo di soluzione fallito

Un primo tentativo con `pd.to_numeric(df['LAT'], errors='coerce')` ha prodotto **2.131.480 valori nulli** (~68% del dataset). Il `coerce` infatti trasforma in NaN tutto ciò che non è convertibile, e siccome `33,9825` non è un numero valido per Python, **tutte le coordinate del primo dataset sono state distrutte**.

### Soluzione corretta

Prima di convertire, normalizziamo il formato sostituendo la virgola con il punto. La sequenza diventa:

1. Forza il tipo a stringa con `astype(str)` (sicuro anche per i float del secondo dataset, che diventano stringhe con punto)
2. Sostituisci `,` con `.` tramite `str.replace`
3. Converti a `float` con `pd.to_numeric(errors='coerce')`

In questo modo entrambi i formati vengono normalizzati e nessun dato valido viene perso.

In [40]:
df['LAT'] = df['LAT'].astype(str).str.replace(',', '.', regex=False)
df['LON'] = df['LON'].astype(str).str.replace(',', '.', regex=False)

df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

print("Tipi:", df[['LAT', 'LON']].dtypes.to_dict())
print(f"Nulli LAT: {df['LAT'].isnull().sum()}")
print(f"Nulli LON: {df['LON'].isnull().sum()}")

Tipi: {'LAT': dtype('float64'), 'LON': dtype('float64')}
Nulli LAT: 0
Nulli LON: 0


In [41]:
print(f"Range LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"Range LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")
print(f"\nValori a (0, 0): {((df['LAT'] == 0) & (df['LON'] == 0)).sum()}")

Range LAT: 0.0000 → 34.7907
Range LON: -118.8279 → 0.0000

Valori a (0, 0): 3148


In [42]:
df.to_parquet('../../data/processed/crimes_merged.parquet')
print("Salvato in data/processed/crimes_merged.parquet")

Salvato in data/processed/crimes_merged.parquet


## 7. Conclusioni — Fase Plan

## Conclusioni — Fase Plan

**Dimensioni dataset**: 3.138.031 righe × 28 colonne (~670 MB).

**Duplicati**: 57.809 record duplicati su DR_NO (~1.8%). 
Verranno ispezionati in EDA per definire la strategia di gestione.

**Valori nulli rilevanti**:
- `Crm Cd 2/3/4`: >93% nulli → eliminate (un crimine principale è sufficiente)
- `Cross Street`: 84% nulli → eliminata (info ridondante con LOCATION + LAT/LON)
- `Weapon Used Cd / Weapon Desc`: 67% nulli → semantica "nessuna arma", mantenute
- `Mocodes`: 12% nulli → mantenuti per analisi di pattern dei modus operandi
- `Vict Sex / Vict Descent`: 11% nulli → ricodificati come "Unknown"

**Anomalie temporali identificate**:
- Anni 2015–2016 mostrano valori anomali (calo nel 2015, picco nel 2016), 
  probabilmente legati alla transizione del sistema di classificazione LAPD 
  (da UCR a NIBRS). Da verificare e documentare nella relazione finale.
- Anno 2024 incompleto (127.567 record): da verificare in EDA il mese 
  di troncamento e gestire di conseguenza nelle analisi temporali.

**Tipi di dato da convertire**:
- `Date Rptd`, `DATE OCC` → datetime
- `TIME OCC` → orario formattato
- `LAT`, `LON` → float (attualmente object, da indagare in cleaning)
- `Vict Age` → gestire valori sentinella (0, negativi)

**Decisioni operative**:
- Periodo di riferimento: 2010–2024 (dataset completo)
- Cleaning completo, non minimale, per supportare future domande di analisi
- Feature engineering posticipata in un notebook dedicato (03_feature_engineering)
- Eventuale sampling (CL 99% / MoE 1%) valutato a posteriori se necessario